# Speech Emotion Recognition (SER) on ViSEC Dataset

This notebook runs the full comparative benchmarking pipeline for Vietnamese Speech Emotion Recognition on the ViSEC dataset.

In [ ]:
# ==================== SER SYSTEM CONFIGURATION ====================
# MODE has 2 options:
# - "demo": Run super fast, automatically load saved benchmark results (benchmark_results_gpu.json)
#           and trained models/checkpoints (best_ecapa_model.pth, dualstream_model/*.pkl).
#           Evaluate (Inference) on the leak-free Test set and plot charts/confusion matrices immediately.
# - "retrain": Retrain all models from scratch (Classical ML, ECAPA-TDNN, DFAT Hybrid Fusion)
#              on the full dataset (5,280 samples) using the shared split_manifest.json.
MODE = "demo" # Change to "retrain" to retrain from scratch

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ==================== INSTALL REQUIRED LIBRARIES ====================
!pip install datasets librosa soundfile transformers torch torchaudio scikit-learn xgboost optuna openai-whisper speechbrain -q


In [ ]:
# ==================== PART 1: LOAD DATA & LEAK-FREE SPLIT ====================
print("📥 LOADING VISEC DATASET AND READING LEAK-FREE MANIFEST...")

import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib
if not any('ipykernel' in arg for arg in sys.argv) and not hasattr(sys, 'ps1'):
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder

# Add root directory to sys.path for importing
sys.path.append(os.path.abspath("."))

# Load original dataset
print("Loading ViSEC dataset...")
dataset = load_dataset("hustep-lab/ViSEC", trust_remote_code=True)
df = dataset['train'].to_pandas()

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['emotion'])
emotion_labels = le.classes_.tolist()
num_labels = len(emotion_labels)

# Read fixed split_manifest.json as the single source of truth
manifest_path = "split_manifest.json"
if not os.path.exists(manifest_path):
    print("split_manifest.json not found, please run generate_splits.py first.")
    raise FileNotFoundError("Missing split_manifest.json")

with open(manifest_path, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

train_idx = manifest['train_indices']
val_idx = manifest['val_indices']
test_idx = manifest['test_indices']

# Assign train/val/test data variables
X_train = df['path'].iloc[train_idx].values
y_train = df['label'].iloc[train_idx].values
X_val = df['path'].iloc[val_idx].values
y_val = df['label'].iloc[val_idx].values
X_test = df['path'].iloc[test_idx].values
y_test = df['label'].iloc[test_idx].values

# Classical ML merges Train + Val
X_trainval = df['path'].iloc[list(train_idx) + list(val_idx)].values
y_trainval = df['label'].iloc[list(train_idx) + list(val_idx)].values

print(f"✓ Manifest loaded: {manifest['total_samples']} samples")
print(f"  - Train: {len(X_train)} samples")
print(f"  - Val:   {len(X_val)} samples")
print(f"  - Test:  {len(X_test)} samples")


## Model 1: SVM with MFCC Features

In [ ]:
# ==================== PART 2: MODEL 1 - SVM with MFCC ====================
print("🤖 MODEL 1: SVM with MFCC Features")

from benchmark_methods_gpu import load_audio, mfcc_feature, evaluate_method
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import time

if MODE == "demo":
    print("✨ DEMO MODE: Loading pre-computed results...")
    with open("benchmark_results_gpu.json", "r", encoding="utf-8") as f:
        bench_res = json.load(f)
    
    res = next(r for r in bench_res['ranked_results'] if r['method'] == 'MFCC+SVM')
    print(f"✓ F1 Score (Weighted): {res['f1_weighted']:.4f}")
    print(f"✓ Latency: {res['latency_ms_per_sample']:.2f} ms/sample")
    
    cm = np.array(res['confusion_matrix'])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=emotion_labels, yticklabels=emotion_labels)
    plt.title("SVM Confusion Matrix (Test Set)")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
else:
    print("🚀 RETRAIN MODE: Training model from scratch...")
    mfcc_train, mfcc_y_train = [], []
    for p, y in zip(X_trainval, y_trainval):
        audio = load_audio(p)
        if audio is not None:
            mfcc_train.append(mfcc_feature(audio))
            mfcc_y_train.append(y)
            
    mfcc_test, mfcc_y_test = [], []
    for p, y in zip(X_test, y_test):
        audio = load_audio(p)
        if audio is not None:
            mfcc_test.append(mfcc_feature(audio))
            mfcc_y_test.append(y)
            
    scaler = StandardScaler()
    mfcc_train = scaler.fit_transform(mfcc_train)
    mfcc_test = scaler.transform(mfcc_test)
    
    clf = SVC(kernel="rbf", C=10.0, gamma="scale", random_state=42)
    res = evaluate_method("MFCC+SVM", clf, mfcc_train, mfcc_y_train, mfcc_test, mfcc_y_test, emotion_labels)
    print(f"✓ F1 Score (Weighted): {res['f1_weighted']:.4f}")
    print(f"✓ Latency: {res['latency_ms_per_sample']:.2f} ms/sample")


## Model 2: Random Forest with MFCC Features

In [ ]:
# ==================== PART 3: MODEL 2 - Random Forest with MFCC ====================
print("🤖 MODEL 2: Random Forest with MFCC Features")

from sklearn.ensemble import RandomForestClassifier

if MODE == "demo":
    print("✨ DEMO MODE: Loading pre-computed results...")
    with open("benchmark_results_gpu.json", "r", encoding="utf-8") as f:
        bench_res = json.load(f)
    
    res = next(r for r in bench_res['ranked_results'] if r['method'] == 'MFCC+RandomForest')
    print(f"✓ F1 Score (Weighted): {res['f1_weighted']:.4f}")
    print(f"✓ Latency: {res['latency_ms_per_sample']:.2f} ms/sample")
else:
    print("🚀 RETRAIN MODE: Training model from scratch...")
    clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    res = evaluate_method("MFCC+RandomForest", clf, mfcc_train, mfcc_y_train, mfcc_test, mfcc_y_test, emotion_labels)
    print(f"✓ F1 Score (Weighted): {res['f1_weighted']:.4f}")
    print(f"✓ Latency: {res['latency_ms_per_sample']:.2f} ms/sample")


## Model 3: XGBoost with MFCC Features

In [ ]:
# ==================== PART 4: MODEL 3 - XGBoost with MFCC ====================
print("🤖 MODEL 3: XGBoost with MFCC Features")

from xgboost import XGBClassifier

if MODE == "demo":
    print("✨ DEMO MODE: Loading pre-computed results...")
    with open("benchmark_results_gpu.json", "r", encoding="utf-8") as f:
        bench_res = json.load(f)
    
    res = next(r for r in bench_res['ranked_results'] if r['method'] == 'MFCC+XGBoost')
    print(f"✓ F1 Score (Weighted): {res['f1_weighted']:.4f}")
    print(f"✓ Latency: {res['latency_ms_per_sample']:.2f} ms/sample")
else:
    print("🚀 RETRAIN MODE: Training model from scratch...")
    clf = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.08,
                        subsample=0.9, colsample_bytree=0.9, random_state=42, eval_metric="mlogloss")
    res = evaluate_method("MFCC+XGBoost", clf, mfcc_train, mfcc_y_train, mfcc_test, mfcc_y_test, emotion_labels)
    print(f"✓ F1 Score (Weighted): {res['f1_weighted']:.4f}")
    print(f"✓ Latency: {res['latency_ms_per_sample']:.2f} ms/sample")


## Model 4: WavLM Baseline

In [ ]:
# ==================== PART 5: MODEL 4 - WavLM Baseline ====================
print("🤖 MODEL 4: WavLM Feature Extractor + SVM / LogReg")

from transformers import AutoFeatureExtractor, AutoModel
import torch
from benchmark_methods_gpu import wavlm_feature

if MODE == "demo":
    print("✨ DEMO MODE: Loading pre-computed results...")
    with open("benchmark_results_gpu.json", "r", encoding="utf-8") as f:
        bench_res = json.load(f)
    
    res = next((r for r in bench_res['ranked_results'] if r['method'] == 'WavLM+SVM'), None)
    if res:
        print(f"✓ WavLM+SVM F1 Score (Weighted): {res['f1_weighted']:.4f}")
        print(f"✓ Latency: {res['latency_ms_per_sample']:.2f} ms/sample")
else:
    print("🚀 RETRAIN MODE: Training model from scratch...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Extracting WavLM embeddings on {device}...")
    extractor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
    wavlm = AutoModel.from_pretrained("microsoft/wavlm-base-plus").to(device).eval()
    
    wavlm_train, wavlm_y_train = [], []
    for p, y in zip(X_trainval, y_trainval):
        audio = load_audio(p)
        if audio is not None:
            wavlm_train.append(wavlm_feature(audio, extractor, wavlm, device))
            wavlm_y_train.append(y)
            
    wavlm_test, wavlm_y_test = [], []
    for p, y in zip(X_test, y_test):
        audio = load_audio(p)
        if audio is not None:
            wavlm_test.append(wavlm_feature(audio, extractor, wavlm, device))
            wavlm_y_test.append(y)
            
    scaler = StandardScaler()
    wavlm_train = scaler.fit_transform(wavlm_train)
    wavlm_test = scaler.transform(wavlm_test)
    
    clf = SVC(kernel="rbf", C=5.0, gamma="scale", random_state=42)
    res = evaluate_method("WavLM+SVM", clf, wavlm_train, wavlm_y_train, wavlm_test, wavlm_y_test, emotion_labels)
    print(f"✓ WavLM+SVM F1 Score (Weighted): {res['f1_weighted']:.4f}")


## Model 5: ECAPA-TDNN

In [ ]:
# ==================== PART 6: ECAPA-TDNN ====================
print("🤖 CREATING AND TRAINING ECAPA-TDNN MODEL")

import sys
sys.path.append(os.path.abspath("./ECAPA"))
from train_emotion_model import prepare_features, AudioFeaturesDataset, collate_fn, train_epoch, evaluate
from predict_emotion import ECAPA_TDNN, EmotionClassifier
from torch.utils.data import DataLoader
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EmotionClassifier(num_labels).to(device)

if MODE == "demo":
    checkpoint_path = "./ECAPA/emotion_model/best_ecapa_model.pth"
    if os.path.exists(checkpoint_path):
        print(f"✨ DEMO MODE: Found checkpoint {checkpoint_path}, loading model...")
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        
        print("Extracting features for Test set...")
        X_test_feat, y_test_clean = prepare_features(X_test, y_test, "Test")
        test_dataset = AudioFeaturesDataset(X_test_feat, y_test_clean)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
        
        test_preds, test_labels = evaluate(model, test_loader, device)
        from sklearn.metrics import f1_score, accuracy_score
        test_f1_weighted = f1_score(test_labels, test_preds, average='weighted')
        print(f"✓ Final test F1 (weighted): {test_f1_weighted:.4f}")
        
        cm = confusion_matrix(test_labels, test_preds)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=emotion_labels, yticklabels=emotion_labels)
        plt.title("ECAPA-TDNN Confusion Matrix")
        plt.show()
    else:
        print("❌ ECAPA-TDNN checkpoint not found. Please switch to MODE = 'retrain'.")
else:
    print("🚀 RETRAIN MODE: Training model from scratch...")
    # Standard training loop from codebase
    print("Extracting features...")
    X_train_feat, y_train_clean = prepare_features(X_train, y_train, "Train")
    X_val_feat, y_val_clean = prepare_features(X_val, y_val, "Val")
    X_test_feat, y_test_clean = prepare_features(X_test, y_test, "Test")

    train_dataset = AudioFeaturesDataset(X_train_feat, y_train_clean)
    val_dataset = AudioFeaturesDataset(X_val_feat, y_val_clean)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003, weight_decay=0.0001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)
    
    num_epochs = 20 # Shortened for notebook
    best_val_f1 = 0
    from sklearn.metrics import f1_score
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, epoch)
        val_preds, val_labels = evaluate(model, val_loader, device)
        val_f1 = f1_score(val_labels, val_preds, average='weighted')
        if epoch >= 2: scheduler.step(val_f1)
        print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Val F1: {val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            os.makedirs('./ECAPA/emotion_model', exist_ok=True)
            torch.save(model.state_dict(), './ECAPA/emotion_model/best_ecapa_model.pth')
            print(f"  ✓ New best val F1: {best_val_f1:.4f} — checkpoint saved")
    
    # Load best model for final evaluation
    if best_val_f1 > 0:
        model.load_state_dict(torch.load('./ECAPA/emotion_model/best_ecapa_model.pth', map_location=device, weights_only=True))
        print(f"
✓ Best model loaded (Val F1: {best_val_f1:.4f})")


## Model Comparison

In [ ]:
# ==================== PART 7: SUMMARY COMPARISON TABLE ====================
print("📊 SUMMARY COMPARISON TABLE")

if os.path.exists("benchmark_results_gpu.json"):
    with open("benchmark_results_gpu.json", "r", encoding="utf-8") as f:
        bench_res = json.load(f)
    
    methods = []
    f1s = []
    lats = []
    accs = []
    for r in bench_res['ranked_results']:
        methods.append(r['method'])
        f1s.append(r['f1_weighted'])
        lats.append(r['latency_ms_per_sample'])
        accs.append(r['accuracy'])
        
    comparison_df = pd.DataFrame({
        'Model': methods,
        'F1 Score': f1s,
        'Latency (ms)': lats,
        'Accuracy': accs
    })
    
    print()
    print(comparison_df.to_string(index=False))
    
    # Plot comparison charts
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5))
    ax1.barh(comparison_df['Model'], comparison_df['F1 Score'], color='skyblue')
    ax1.set_xlabel('F1 Score')
    ax1.set_title('F1 Score Comparison')
    ax1.set_xlim([0, 1])
    
    ax2.barh(comparison_df['Model'], comparison_df['Latency (ms)'], color='salmon')
    ax2.set_xlabel('Latency (ms/sample)')
    ax2.set_title('Latency Comparison')
    
    ax3.barh(comparison_df['Model'], comparison_df['Accuracy'], color='lime')
    ax3.set_xlabel('Accuracy')
    ax3.set_title('Accuracy Comparison')
    
    plt.tight_layout()
    plt.show()
else:
    print("Please run the full benchmark to display the comparison table.")


## Model 6: DFAT Hybrid Fusion (Proposed Method)

In [ ]:
# ==================== PART 8: DFAT HYBRID FUSION ====================
print("🤖 DFAT HYBRID FUSION: WavLM + Whisper")

import sys
sys.path.append(os.path.abspath("./DFAT Hybrid Fusion"))
from train_dualstream import extract_features_for_split
from transformers import AutoFeatureExtractor, AutoModel, WhisperProcessor, WhisperForConditionalGeneration
import pickle

model_dir = "./DFAT Hybrid Fusion/dualstream_model"

if MODE == "demo":
    metadata_path = os.path.join(model_dir, "metadata.json")
    if os.path.exists(metadata_path):
        print(f"✨ DEMO MODE: Found model in {model_dir}, loading model...")
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)
        
        print(f"✓ Ensemble F1 Score (Weighted): {metadata['test_f1_weighted']:.4f}")
        
        cm = np.array(metadata['confusion_matrix'])
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=emotion_labels, yticklabels=emotion_labels)
        plt.title("DFAT Hybrid Fusion Confusion Matrix")
        plt.show()
    else:
        print("❌ DFAT model not found. Please switch to MODE = 'retrain'.")
else:
    print("🚀 RETRAIN MODE: Training model from scratch...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Loading WavLM & Whisper...")
    wavlm_processor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
    wavlm_model = AutoModel.from_pretrained("microsoft/wavlm-base-plus").to(device).eval()
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device).eval()
    
    print("Extracting Test features...")
    test_fused, test_labels_ext = extract_features_for_split(
        X_test, y_test, wavlm_model, wavlm_processor, whisper_model, whisper_processor, device, "Test"
    )
    # Shortened training process for notebook, focusing on existing code in the python script
    print("Please refer to train_dualstream.py to see the full leak-free Optuna tuning process.")
